# 03 · Honest improvements — pre-registered experiments
**Group 2 · ZAS Data Science Regional Hackathon 2026**

Goal: improve the model **without** overfitting to the numbers we report.

### The problem with our earlier experiments
Some design choices (the n ≥ 25 class-mix rule, the two-break features) were checked on the **same 10 Amsterdam draws we report**. That is selection bias: the reported score is slightly optimistic.

### Protocol used here (fixed before running)
1. Amsterdam is split by whole **2 km blocks** into **DEV** (half the blocks) and **TEST** (other half), random seed 2026.
2. Every candidate change is judged on **DEV only**: support labels drawn from DEV, scored on the rest of DEV, **new draw seeds 100–109** (never used for reporting).
3. Madrid-side changes are judged on **Madrid spatial-block CV only**.
4. **Adoption rule** (paired over all n × draws): mean gain > **2 standard errors** *and* the score on pixels ≥ 1 km from any label (**far pixels**) does not drop by more than 0.005.
5. Smoothing stays only if it does not hurt far pixels by more than 0.005.
6. The combined configuration is fixed, then scored **once** on **TEST** (support from DEV, query in TEST — labels are never inside the test region), draws 0–9.

### Candidates (each targets a known weakness)
| ID | Change | Weakness it targets |
|---|---|---|
| E1 | Label-noise weights: Madrid pixels within ±2 years of a class boundary count 0.5 | Class 1/2 confusion, the 1960 label pile |
| E2 | BBSE class-mix estimate (Lipton et al., ICML 2018) instead of EM | EM fails without labels / at n = 5 |
| E3 | Agreement filter for pseudo-labels (recalibrated and fine-tuned models must agree) | self-training confirming its own mistakes |
| E4 | Weight of Amsterdam labels in fine-tuning: 10 % / 50 % (current 25 %) | an untuned setting |
| E5 | Smoothing off | reliance on neighbours (far pixels) |

## 1 · Shared pipeline (identical to the submission notebook: cleaning, 321 features, scaling)

In [1]:
import importlib, subprocess, sys
for pkg in ['lightgbm', 'xgboost', 'catboost', 'pyarrow']:          # install anything the platform is missing
    try: importlib.import_module(pkg)
    except ImportError: subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', pkg], check=False)

import os, gc, json, time, warnings, joblib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import uniform_filter
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix
import lightgbm as lgb, xgboost as xgb, catboost as cb
warnings.filterwarnings('ignore')

DATA_DIR = next(Path(p) for p in ['data', '../data', '.'] if (Path(p) / 'madrid_train.parquet').exists())
OUT = Path('exp03_outputs'); OUT.mkdir(exist_ok=True)
SEED, N_JOBS = 42, os.cpu_count()
CITIES = ['Madrid', 'Amsterdam']; FILES = {'Madrid': 'madrid_train.parquet', 'Amsterdam': 'amsterdam_data.parquet'}
BANDS = ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']; PID = ['px_key', 'py_key']
YEARS = np.arange(1984, 2026); EARLY, LATE = YEARS <= 2003, YEARS >= 2004
CLASSES = np.array([1, 2, 3, 4]); CLASS_EVENTS = {'Madrid': 1960, 'Amsterdam': 1945}
N_SHOTS, N_DRAWS = [5, 25, 50, 100, 200], 10
SMOKE = os.environ.get('SMOKE') == '1'          # quick self-test only (15 % of pixels, 1 draw); off by default
if SMOKE: N_DRAWS = 1
def mf1(a, b): return f1_score(a, b, average='macro')
def log(m): print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)

def _gpu():
    try: return subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, timeout=20).stdout.strip()
    except Exception: return ''
XGB_DEVICE, CAT_TASK = 'cpu', 'CPU'
if _gpu():
    try: xgb.XGBClassifier(n_estimators=2, device='cuda').fit(np.random.rand(50, 3), np.random.randint(0, 2, 50)); XGB_DEVICE = 'cuda'
    except Exception: pass
    try: cb.CatBoostClassifier(iterations=2, task_type='GPU', verbose=0).fit(np.random.rand(50, 3), np.random.randint(0, 2, 50)); CAT_TASK = 'GPU'
    except Exception: pass
log(f"data: {DATA_DIR.resolve()} | {N_JOBS} CPU threads | XGBoost {XGB_DEVICE} | CatBoost {CAT_TASK}")

[18:21:03] data: . | 8 CPU threads | XGBoost cpu | CatBoost CPU


In [2]:
CUM = np.array([31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334])
def to_long(df):
    parts = []
    for s in (1, 2, 3):
        ren = {f'{b}_{s}': b for b in BANDS}
        ren.update({f'QA_PIXEL_{s}': 'QA_PIXEL', f'QA_RADSAT_{s}': 'QA_RADSAT', f'doy_{s}': 'doy', f'scene_id_{s}': 'scene_id', f'qa_valid_{s}': 'qa_valid'})
        parts.append(df[PID + ['year'] + list(ren)].rename(columns=ren))
    o = pd.concat(parts, ignore_index=True); o[BANDS] = o[BANDS].astype('float32')
    o['month'] = np.searchsorted(CUM, o.doy.to_numpy(), side='left') + 1
    o['cirrus'] = (o.QA_PIXEL.to_numpy().astype(np.int64) >> 14) & 3
    return o

def find_bad_scenes(o):
    v = o[o.qa_valid.astype(bool)]
    t = v.groupby('scene_id', observed=True).Blue.agg(med='median', frac=lambda x: (x > 15000).mean())
    return set(t.index[(t.med > 12000) | (t.frac > .05)])

def clean_and_composite(raw, bad_scenes):
    o = to_long(raw)
    bad = (~o.qa_valid.astype(bool)) | o.scene_id.isin(bad_scenes) | (o.QA_RADSAT > 0) | (o[BANDS] == 0).any(axis=1) | (o.cirrus == 3) | (o.Blue > 15000)
    o = o[~bad.values]
    keys = PID + ['year']; warm = o.month.between(5, 9)
    allm = o.groupby(keys)[BANDS].median(); warmm = o[warm].groupby(keys)[BANDS].median().reindex(allm.index)
    use = warmm.notna().all(axis=1); comp = warmm.where(use, allm)
    comp['n_photos'] = o.groupby(keys).size(); comp['fallback'] = (~use).astype(float)
    return comp.reset_index()

def pixel_table(raw, city=None):
    agg = {'coverage': ('coverage', 'first')}
    if 'weighted_mean_year' in raw: agg['wmy'] = ('weighted_mean_year', 'first')
    p = raw.groupby(PID).agg(**agg).reset_index().sort_values(PID).reset_index(drop=True)
    if city and 'wmy' in p:
        p['age_class'] = pd.cut(p.wmy, [-np.inf, CLASS_EVENTS[city], 1984, 2004, np.inf], labels=[1, 2, 3, 4]).astype(int)
    return p

RAW, YEARLY, PIX, BAD = {}, {}, {}, set()
for c in CITIES:
    RAW[c] = pd.read_parquet(DATA_DIR / FILES[c])
    if SMOKE: RAW[c] = RAW[c].merge(RAW[c][PID].drop_duplicates().sample(frac=.15, random_state=SEED), on=PID)
    BAD |= find_bad_scenes(to_long(RAW[c]))
def starter_flat(raw):                    # the organisers' starter view: first clear photo per year, Blue <= 15 000
    out = raw[PID + ['year']].copy()
    for b in BANDS:
        col = np.full(len(raw), np.nan)
        for s in (1, 2, 3):
            col = np.where(np.isnan(col) & raw[f'qa_valid_{s}'].fillna(False).to_numpy(bool), raw[f'{b}_{s}'].to_numpy(), col)
        out[b] = col
    out = out.dropna(subset=BANDS)
    return out[out.Blue <= 15000]
STARTER_FLAT = {}
for c in CITIES:
    PIX[c] = pixel_table(RAW[c], c)
    YEARLY[c] = clean_and_composite(RAW[c], BAD)
    STARTER_FLAT[c] = starter_flat(RAW[c])
    log(f"{c}: {len(PIX[c]):,} pixels, {len(YEARLY[c]):,} clean pixel-years")
log(f"broken scenes removed: {sorted(BAD)}")
del RAW; gc.collect()

[18:21:20] Madrid: 76,263 pixels, 3,147,969 clean pixel-years


[18:21:23] Amsterdam: 25,992 pixels, 1,058,169 clean pixel-years


[18:21:23] broken scenes removed: ['LE07_L2SP_198023_20020417_20200916_02_T2', 'LT05_L2SP_199023_19961109_20211124_02_T1', 'LT05_L2SP_201032_19951207_20211123_02_T2']


0

In [3]:
TC = {'TCB': [0.3029, 0.2786, 0.4733, 0.5599, 0.5080, 0.1872], 'TCG': [-0.2941, -0.2430, -0.5424, 0.7276, 0.0713, -0.1608],
      'TCW': [0.1511, 0.1973, 0.3283, 0.3407, -0.7117, -0.4559]}
IDX5 = ['NDVI', 'NDBI', 'UI', 'MNDWI', 'BSI']; IDXN = ['NBR', 'NDTI', 'TCB', 'TCG', 'TCW']
K = ['NDVI', 'NDBI', 'MNDWI', 'BSI', 'NBR', 'NDTI', 'TCB', 'TCG', 'TCW', 'SWIR1', 'NIR']
nmean = lambda M: np.nanmean(M, 1); nstd = lambda M: np.nanstd(M, 1, ddof=1); nmed = lambda M: np.nanmedian(M, 1)

def make_cube(yr, pix):
    wide = yr.set_index(PID + ['year'])[BANDS].unstack('year').reindex(pd.MultiIndex.from_frame(pix[PID]))
    R = {b: wide[b].reindex(columns=YEARS).to_numpy(np.float32) for b in BANDS}
    B, G, Rd, N, S1, S2 = (R[b] for b in BANDS); e = 1e-6
    R.update({'NDVI': (N - Rd) / (N + Rd + e), 'NDBI': (S1 - N) / (S1 + N + e), 'UI': (S2 - N) / (S2 + N + e), 'MNDWI': (G - S1) / (G + S1 + e),
              'BSI': ((S1 + Rd) - (N + B)) / ((S1 + Rd) + (N + B) + e), 'NBR': (N - S2) / (N + S2 + e), 'NDTI': (S1 - S2) / (S1 + S2 + e)})
    refl = [R[b] * 2.75e-5 - 0.2 for b in BANDS]
    for k, w in TC.items(): R[k] = sum(wi * r for wi, r in zip(w, refl)).astype(np.float32)
    F = {k: pd.DataFrame(v.T).interpolate(limit_direction='both').to_numpy(np.float32).T for k, v in R.items()}
    return R, F, ~np.isnan(R['Blue'])

def slope(M, yrs):
    x = np.where(np.isnan(M), np.nan, yrs[None, :].astype(np.float32)); xm, ym = np.nanmean(x, 1, keepdims=True), np.nanmean(M, 1, keepdims=True)
    num, den = np.nansum((x - xm) * (M - ym), 1), np.nansum((x - xm) ** 2, 1)
    return np.where(den > 0, num / np.where(den > 0, den, 1), np.nan)

def step_break(Y):
    T = Y.shape[1]; k = np.arange(1, T); cs = np.cumsum(Y, 1, dtype=np.float64)[:, :-1]; tot = Y.sum(1, keepdims=True, dtype=np.float64)
    bef, aft = cs / k, (tot - cs) / (T - k); gain = k * (T - k) / T * (aft - bef) ** 2; b = gain.argmax(1); i = np.arange(len(Y))
    return YEARS[b + 1], (aft - bef)[i, b], gain[i, b] / (Y.var(1, dtype=np.float64) * T + 1e-12)

def two_break(Y):
    Y = Y.astype(np.float64); n, T = Y.shape; cs = np.concatenate([np.zeros((n, 1)), np.cumsum(Y, 1)], 1)
    tss = ((Y - Y.mean(1, keepdims=True)) ** 2).sum(1) + 1e-12; base = cs[:, T] ** 2 / T
    best = np.full(n, -np.inf); bi = np.zeros(n, int); bj = np.zeros(n, int)
    for i in range(2, T - 1):
        js = np.arange(i + 2, T - 1)
        if not len(js): continue
        g = cs[:, [i]] ** 2 / i + (cs[:, js] - cs[:, [i]]) ** 2 / (js - i) + (cs[:, [T]] - cs[:, js]) ** 2 / (T - js) - base[:, None]
        k = g.argmax(1); gv = g[np.arange(n), k]; u = gv > best; best[u] = gv[u]; bi[u] = i; bj[u] = js[k[u]]
    r = np.arange(n); l1, l2, l3 = cs[r, bi] / bi, (cs[r, bj] - cs[r, bi]) / (bj - bi), (cs[r, T] - cs[r, bj]) / (T - bj)
    return YEARS[bi], YEARS[bj], l2 - l1, l3 - l2, best / tss

def base_features(yr, pix):
    R, F, m = make_cube(yr, pix); o = {}
    for ch in BANDS + IDX5: o[f'{ch}_mean'] = nmean(R[ch]); o[f'{ch}_std'] = nstd(R[ch])                       # S
    for b in BANDS:
        o[f'{b}_early_mean'] = nmean(R[b][:, EARLY]); o[f'{b}_early_std'] = nstd(R[b][:, EARLY])
        o[f'{b}_late_mean'] = nmean(R[b][:, LATE]); o[f'{b}_late_std'] = nstd(R[b][:, LATE])
        d = np.diff(F[b], axis=1); o[f'd_{b}_mean'] = d.mean(1); o[f'd_{b}_std'] = d.std(1, ddof=1)
    for ch in IDXN:                                                                                               # A
        o[f'{ch}_mean'] = nmean(R[ch]); o[f'{ch}_std'] = nstd(R[ch]); o[f'{ch}_early_mean'] = nmean(R[ch][:, EARLY]); o[f'{ch}_late_mean'] = nmean(R[ch][:, LATE])
    for ch in K:                                                                                                  # B
        q = {p: np.nanpercentile(R[ch], p, axis=1) for p in (10, 25, 50, 75, 90)}
        o[f'{ch}_med'] = q[50]; o[f'{ch}_iqr'] = q[75] - q[25]; o[f'{ch}_p10'] = q[10]; o[f'{ch}_p90'] = q[90]
        o[f'{ch}_slope'] = slope(R[ch], YEARS); o[f'{ch}_slope_early'] = slope(R[ch][:, EARLY], YEARS[EARLY]); o[f'{ch}_slope_late'] = slope(R[ch][:, LATE], YEARS[LATE])
        o[f'{ch}_volatility'] = nmed(np.abs(np.diff(R[ch], axis=1)))
    for ch in ['NDVI', 'NDBI', 'BSI', 'TCB', 'SWIR1']:                                                            # C
        for a in range(1984, 2026, 6): o[f'{ch}_w{a}'] = nmed(R[ch][:, (YEARS >= a) & (YEARS <= a + 5)])
    S_, Y_ = [], []                                                                                               # D
    for ch in ['NDVI', 'NDBI', 'BSI', 'SWIR1', 'TCB', 'TCW', 'NBR']:
        y_, s_, st = step_break(F[ch]); o[f'{ch}_brk_year'] = y_; o[f'{ch}_brk_size'] = s_; o[f'{ch}_brk_strength'] = st
        o[f'{ch}_brk_after2004'] = (y_ >= 2004) * st; S_.append(st); Y_.append(y_)
    S_, Y_ = np.vstack(S_), np.vstack(Y_); o['brk_year_best'] = Y_[S_.argmax(0), np.arange(S_.shape[1])]; o['brk_strength_max'] = S_.max(0)
    for ch in ['NDVI', 'NDBI', 'SWIR1']:
        d = np.diff(F[ch], axis=1); j = np.abs(d).argmax(1); o[f'{ch}_jump_year'] = YEARS[j + 1]; o[f'{ch}_jump_size'] = d[np.arange(len(d)), j]
    for ch in K: o[f'{ch}_late_minus_early'] = nmed(R[ch][:, LATE]) - nmed(R[ch][:, EARLY])                      # E
    meds = {b: nmed(R[b]) for b in BANDS}; tot = sum(meds.values())
    for b in BANDS: o[f'{b}_late_over_early'] = nmed(R[b][:, LATE]) / nmed(R[b][:, EARLY]); o[f'{b}_shape'] = meds[b] / tot
    yr2 = yr.groupby(PID).agg(ppy=('n_photos', 'mean'), fb=('fallback', 'mean')).reindex(pd.MultiIndex.from_frame(pix[PID]))
    run = np.zeros(len(m)); best = np.zeros(len(m))
    for t in range(m.shape[1]): run = (run + 1) * (~m[:, t]); best = np.maximum(best, run)
    o.update({'coverage': pix.coverage.values, 'frac_filled': 1 - m.mean(1), 'longest_gap': best,                # F
              'photos_per_year': yr2.ppy.values, 'frac_fallback': yr2.fb.values})
    for ch in ['NDVI', 'NDBI', 'TCB']:                                                                            # H
        y1, y2, d1, d2, st = two_break(F[ch]); _, _, st1 = step_break(F[ch])
        o.update({f'{ch}_brk2_y1': y1, f'{ch}_brk2_y2': y2, f'{ch}_brk2_d1': d1, f'{ch}_brk2_d2': d2, f'{ch}_brk2_strength': st, f'{ch}_brk2_extra': st - st1})
    for ch, sg in [('NDVI', -1), ('NDBI', 1)]:
        ma = pd.DataFrame(F[ch].T).rolling(3, center=True, min_periods=1).mean().to_numpy().T; d = sg * np.diff(ma, axis=1); j = d.argmax(1)
        o[f'{ch}_steep_year'] = YEARS[j + 1]; o[f'{ch}_steep_size'] = d[np.arange(len(d)), j]
    return pd.DataFrame(o)

def neigh(pix, v, size, stat='mean'):
    x = (pix.px_key - pix.px_key.min()).to_numpy(); y_ = (pix.py_key - pix.py_key.min()).to_numpy(); H, W = y_.max() + 1, x.max() + 1; ok = ~np.isnan(v)
    V, M, V2 = np.zeros((H, W)), np.zeros((H, W)), np.zeros((H, W)); V[y_[ok], x[ok]] = v[ok]; M[y_[ok], x[ok]] = 1; V2[y_[ok], x[ok]] = v[ok] ** 2
    n = size ** 2; s = uniform_filter(V, size, mode='constant') * n - V; s2 = uniform_filter(V2, size, mode='constant') * n - V2
    c = (uniform_filter(M, size, mode='constant') * n - M)[y_, x]; mean = np.where(c > .5, s[y_, x] / np.maximum(c, 1), np.nan)
    if stat == 'mean': return mean
    return np.sqrt(np.clip(np.where(c > 1.5, s2[y_, x] / np.maximum(c, 1) - mean ** 2, np.nan), 0, None))

GKEYS = ['NDVI_med', 'NDBI_med', 'TCB_med', 'SWIR1_med', 'NDVI_late_minus_early', 'NDVI_brk_strength', 'NDVI_brk_year', 'NDBI_brk_size', 'coverage']
def spatial_features(pix, base):
    o = {}; x = (pix.px_key - pix.px_key.min()).to_numpy(); y_ = (pix.py_key - pix.py_key.min()).to_numpy()
    Mk = np.zeros((y_.max() + 1, x.max() + 1)); Mk[y_, x] = 1
    for f in GKEYS:
        for s in (3, 7, 15): o[f'nb{s}_{f}'] = neigh(pix, base[f].to_numpy(np.float64), s)
    for s in (3, 7, 15): o[f'density{s}'] = (uniform_filter(Mk, s, mode='constant') * s ** 2 - 1)[y_, x] / (s ** 2 - 1)
    for f in ['NDVI_med', 'NDVI_brk_year']: o[f'nbstd7_{f}'] = neigh(pix, base[f].to_numpy(np.float64), 7, 'std')
    for f in ['NDVI_med', 'TCB_med']: o[f'diff7_{f}'] = base[f].to_numpy() - o[f'nb7_{f}']
    return pd.DataFrame(o)

BASE, FEATS = {}, {}
for c in CITIES:
    t = time.time(); BASE[c] = base_features(YEARLY[c], PIX[c])
    FEATS[c] = pd.concat([BASE[c], spatial_features(PIX[c], BASE[c])], axis=1)
    log(f"{c}: {FEATS[c].shape[1]} features ({time.time()-t:.0f}s)")
y = {c: PIX[c].age_class.to_numpy() for c in CITIES}; ym, ya = y['Madrid'], y['Amsterdam']

[18:24:48] Madrid: 321 features (206s)


[18:26:05] Amsterdam: 321 features (77s)


In [4]:
def fit_scaler(X): mu = np.nanmean(X, 0); sd = np.nanstd(X, 0); sd[~(sd > 0)] = 1; return mu, sd
def apply_scaler(X, st): return ((X - st[0]) / st[1]).astype(np.float32)
ALL = list(FEATS['Madrid'].columns)
cw = len(ym) / (4 * np.bincount(ym, minlength=5)[1:]); W_MAD = PIX['Madrid'].coverage.to_numpy() * cw[ym - 1]; W_MAD /= W_MAD.mean()
LGB = dict(n_estimators=400, learning_rate=0.05, num_leaves=63, colsample_bytree=0.5, subsample=0.8, subsample_freq=1, min_child_samples=20,
           n_jobs=N_JOBS, random_state=SEED, verbose=-1)
sel = lgb.LGBMClassifier(**LGB, importance_type='gain').fit(apply_scaler(FEATS['Madrid'][ALL].to_numpy(np.float64), fit_scaler(FEATS['Madrid'][ALL].to_numpy(np.float64))), ym, sample_weight=W_MAD)
RICH = list(pd.Series(sel.feature_importances_, index=ALL).sort_values(ascending=False).index[:120])
G_S = [f'{ch}_{s}' for ch in BANDS + IDX5 for s in ('mean', 'std')] + \
      [f'{p}{b}_{s}' for b in BANDS for p, s in [('', 'early_mean'), ('', 'early_std'), ('', 'late_mean'), ('', 'late_std'), ('d_', 'mean'), ('d_', 'std')]]
G_D = [f'{ch}_brk_{s}' for ch in ['NDVI', 'NDBI', 'BSI', 'SWIR1', 'TCB', 'TCW', 'NBR'] for s in ('year', 'size', 'strength', 'after2004')] + \
      ['brk_year_best', 'brk_strength_max'] + [f'{ch}_jump_{s}' for ch in ['NDVI', 'NDBI', 'SWIR1'] for s in ('year', 'size')]
G_H = [f'{ch}_brk2_{s}' for ch in ['NDVI', 'NDBI', 'TCB'] for s in ('y1', 'y2', 'd1', 'd2', 'strength', 'extra')] + \
      [f'{ch}_steep_{s}' for ch in ['NDVI', 'NDBI'] for s in ('year', 'size')]
TRANSFER = G_S + G_D + G_H + [f'{ch}_{s}' for ch in ['NDVI', 'NDBI', 'MNDWI', 'BSI', 'NBR', 'NDTI', 'TCG', 'TCW'] for s in ['med', 'iqr', 'slope']]
assert len(TRANSFER) == 140 and set(TRANSFER) <= set(ALL)
SCALER = {(c, k): fit_scaler(FEATS[c][cols].to_numpy(np.float64)) for c in CITIES for k, cols in [('RICH', RICH), ('TRANSFER', TRANSFER)]}
X = {(c, k): apply_scaler(FEATS[c][cols].to_numpy(np.float64), SCALER[(c, k)]) for c in CITIES for k, cols in [('RICH', RICH), ('TRANSFER', TRANSFER)]}
log(f"RICH = {len(RICH)} features, TRANSFER = {len(TRANSFER)} features")

[18:26:53] RICH = 120 features, TRANSFER = 140 features


## 2 · E1 on Madrid — label-noise weights (judged on Madrid spatial-block CV only)
Pixels whose weighted mean year lies within ±2 years of a class boundary (1960, 1984, 2004) are the most likely to be mislabelled (mixed buildings, rounding to 1960). They keep half weight. The metric is still computed on **all** pixels with their original labels.

In [5]:
from scipy.spatial import cKDTree
if SMOKE: N_SHOTS = [5, 25]
BLOCKS = (PIX['Madrid'].px_key // 67).astype(str) + '_' + (PIX['Madrid'].py_key // 67).astype(str)
SPATIAL_FOLDS = list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(ym)), ym, BLOCKS))
def make(name):
    if name == 'LightGBM': return lgb.LGBMClassifier(**LGB)
    if name == 'XGBoost':  return xgb.XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.08, subsample=0.8, colsample_bytree=0.5, tree_method='hist',
                                                    device=XGB_DEVICE, n_jobs=N_JOBS, random_state=SEED, eval_metric='mlogloss')
    if name == 'RandomForest': return RandomForestClassifier(n_estimators=300, min_samples_leaf=2, max_features='sqrt', n_jobs=N_JOBS, random_state=SEED)
    if name == 'CatBoost': return cb.CatBoostClassifier(iterations=400, depth=6, learning_rate=0.08, thread_count=N_JOBS, random_seed=SEED, verbose=0, task_type=CAT_TASK)
class Ensemble:
    MEMBERS = ['LightGBM', 'XGBoost', 'RandomForest']
    def fit(self, Xm, yv, w):
        self.models = []
        for n in self.MEMBERS:
            m = make(n); Xi = np.nan_to_num(Xm) if n == 'RandomForest' else Xm
            m.fit(Xi, yv - 1 if n == 'XGBoost' else yv, sample_weight=w); self.models.append((n, m))
        return self
    def predict_proba(self, Xm):
        return np.mean([m.predict_proba(np.nan_to_num(Xm) if n == 'RandomForest' else Xm) for n, m in self.models], axis=0)

wmy = PIX['Madrid'].wmy.to_numpy(); near = np.zeros(len(ym), bool)
for b in (1960, 1984, 2004): near |= np.abs(wmy - b) <= 2
W_NOISE = W_MAD * np.where(near, .5, 1.); W_NOISE /= W_NOISE.mean()
log(f"pixels within ±2 years of a boundary: {near.mean():.1%}")
XR = X[('Madrid', 'RICH')]
def cv_w(w, label):
    P = np.zeros((len(ym), 4)); sc = []
    for f, (tr, te) in enumerate(SPATIAL_FOLDS):
        P[te] = Ensemble().fit(XR[tr], ym[tr], w[tr]).predict_proba(XR[te]); sc.append(mf1(ym[te], CLASSES[P[te].argmax(1)]))
    log(f"  {label}: {np.mean(sc):.4f} ± {np.std(sc):.4f}  folds {np.round(sc, 3)}")
    return np.array(sc), P
sc_e0, OOF = cv_w(W_MAD, 'E0 current weights   ')
sc_e1, OOF_E1 = cv_w(W_NOISE, 'E1 label-noise weights')
d = sc_e1 - sc_e0; se = d.std(ddof=1) / np.sqrt(len(d))
ADOPT_E1_MADRID = bool(d.mean() > 2 * se)
log(f"E1 Madrid paired gain {d.mean():+.4f} (SE {se:.4f}) -> {'ADOPT' if ADOPT_E1_MADRID else 'reject'}")

[18:26:53] pixels within ±2 years of a boundary: 20.7%


[18:35:19]   E0 current weights   : 0.6509 ± 0.0254  folds [0.623 0.662 0.62  0.664 0.686]


[18:44:39]   E1 label-noise weights: 0.6515 ± 0.0258  folds [0.619 0.666 0.624 0.663 0.685]


[18:44:39] E1 Madrid paired gain +0.0007 (SE 0.0014) -> reject


## 3 · Amsterdam DEV / TEST split and the transfer building blocks
Smoothing is tuned on Madrid OOF exactly as before. The BBSE class-mix estimate uses only **Madrid labels** (confusion matrix of out-of-fold predictions) and **unlabelled** Amsterdam predictions.

In [6]:
pa = PIX['Amsterdam']; XY = pa[PID].to_numpy()
ablk = ((pa.px_key // 67).astype(str) + '_' + (pa.py_key // 67).astype(str)).to_numpy()
ub = np.unique(ablk); DEV_BLOCKS = set(np.random.default_rng(2026).choice(ub, len(ub) // 2, replace=False))
DEV = np.isin(ablk, list(DEV_BLOCKS)); TEST = ~DEV
display(pd.DataFrame({'DEV': np.bincount(ya[DEV], minlength=5)[1:], 'TEST': np.bincount(ya[TEST], minlength=5)[1:]}, index=[f'C{k}' for k in CLASSES]).T)

def neigh_smooth(P, pix, alpha, size):
    if alpha >= 1: return P
    N = np.column_stack([neigh(pix, P[:, k].astype(np.float64), size) for k in range(4)]); N = np.where(np.isnan(N), P, N)
    S = alpha * P + (1 - alpha) * N; return S / S.sum(1, keepdims=True)
grid = [(a, s, mf1(ym, CLASSES[neigh_smooth(OOF, PIX['Madrid'], a, s).argmax(1)])) for s in (3, 5, 7, 11, 15) for a in (1.0, .7, .5, .3, .15)]
SMOOTH = dict(zip(['alpha', 'size'], max(grid, key=lambda g: g[2])[:2])); log(f"smoothing tuned on Madrid OOF: {SMOOTH}")

XA, XM = X[('Amsterdam', 'TRANSFER')], X[('Madrid', 'TRANSFER')]
WTS = {'base': W_MAD, 'noise': W_NOISE}; LOGP, PRIOR_BBSE = {}, {}
p_src = np.bincount(ym, minlength=5)[1:] / len(ym)
for k, w in WTS.items():
    LOGP[k] = np.log(make('CatBoost').fit(XM, ym, sample_weight=w).predict_proba(XA) + 1e-6)
    oof = np.zeros((len(ym), 4))
    for tr, te in SPATIAL_FOLDS: oof[te] = make('CatBoost').fit(XM[tr], ym[tr], sample_weight=w[tr]).predict_proba(XM[te])
    pred = oof.argmax(1); C = np.array([[np.mean((pred == i) & (ym == j + 1)) for j in range(4)] for i in range(4)])   # joint P(pred=i, y=j) in Madrid
    mu = np.bincount(LOGP[k].argmax(1), minlength=4) / len(XA)                                                        # predicted mix in Amsterdam (no labels)
    wt = np.clip(np.linalg.lstsq(C, mu, rcond=None)[0], 0, None); q = wt * p_src; PRIOR_BBSE[k] = q / q.sum()
    log(f"{k}: zero-shot {mf1(ya, CLASSES[LOGP[k].argmax(1)]):.3f} | BBSE Amsterdam class mix {np.round(PRIOR_BBSE[k], 3)}")
log(f"(for information only, never used: true Amsterdam mix {np.round(np.bincount(ya, minlength=5)[1:] / len(ya), 3)})")

def em_prior(P, prior=np.full(4, .25), iters=200, tol=1e-6):
    pi = prior.copy()
    for _ in range(iters):
        Q = P * (pi / prior); Q /= Q.sum(1, keepdims=True); new = Q.mean(0)
        if np.abs(new - pi).max() < tol: break
        pi = new
    return pi

def self_train(pr, pf, s, ys, seed, agree):
    anchor = (pr + pf) / 2; pool = np.setdiff1d(np.arange(len(XA)), s); pred = anchor[pool].argmax(1)
    ok = (pr[pool].argmax(1) == pf[pool].argmax(1)) if agree else np.ones(len(pool), bool)
    idx, lab, wt = [s], [ys], [np.full(len(s), 3.0)]
    for k in range(4):
        cand = pool[(pred == k) & ok]; top = cand[np.argsort(-anchor[cand, k])[:500]]; idx.append(top); lab.append(np.full(len(top), k + 1)); wt.append(anchor[top, k])
    st = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=15, min_child_samples=10, colsample_bytree=0.7, n_jobs=N_JOBS,
                            random_state=seed, verbose=-1).fit(XA[np.concatenate(idx)], np.concatenate(lab), sample_weight=np.concatenate(wt))
    return st.predict_proba(XA)

def get_prior(kind, n, P, w):
    if kind == 'em25':  return em_prior(P) if n >= 25 else None
    if kind == 'em_all': return em_prior(P)
    if kind == 'bbse_all': return PRIOR_BBSE[w]
    if kind == 'bbse25': return PRIOR_BBSE[w] if n >= 25 else None
    return None

def run_configs(cfgs, pool_mask, query_mask, seeds, tag):
    rows = []
    for n in N_SHOTS:
        t0 = time.time()
        for seed in seeds:
            rng = np.random.default_rng(seed)
            s = np.concatenate([rng.choice(np.where((ya == k) & pool_mask)[0], n, replace=False) for k in CLASSES]); ys = ya[s]
            q = np.setdiff1d(np.where(query_mask)[0], s); far = q[cKDTree(XY[s]).query(XY[q])[0] >= 33]
            FT, ST = {}, {}
            for name, c in cfgs.items():
                key = (c['w'], c['wfrac'])
                if key not in FT:
                    lp, wm = LOGP[c['w']], WTS[c['w']]
                    pr = LogisticRegression(C=1.0, max_iter=3000).fit(lp[s], ys).predict_proba(lp)
                    ws = np.full(len(s), c['wfrac'] * wm.sum() / len(s))
                    pf = lgb.LGBMClassifier(**dict(LGB, n_estimators=300, random_state=seed)).fit(np.vstack([XM, XA[s]]), np.r_[ym, ys], sample_weight=np.r_[wm, ws]).predict_proba(XA)
                    FT[key] = (pr, pf)
                pr, pf = FT[key]; skey = key + (c['agree'],)
                if skey not in ST: ST[skey] = self_train(pr, pf, s, ys, seed, c['agree'])
                P = (pr + pf + ST[skey]) / 3
                prior = get_prior(c['prior'], n, P, c['w'])
                if prior is not None: P = P * (prior / .25); P /= P.sum(1, keepdims=True)
                if c['smooth']: P = neigh_smooth(P, pa, **SMOOTH)
                yh = CLASSES[P.argmax(1)]
                rows.append({'tag': tag, 'cfg': name, 'n': n, 'seed': seed, 'f1': mf1(ya[q], yh[q]),
                             'f1_far': mf1(ya[far], yh[far]) if len(far) >= 100 and len(np.unique(ya[far])) == 4 else np.nan, 'n_far': len(far)})
        log(f"  [{tag}] n = {n:3d} done ({time.time() - t0:.0f}s)")
    return pd.DataFrame(rows)

E0 = dict(w='base', wfrac=.25, agree=False, prior='em25', smooth=True)
def V(**kw): return dict(E0, **kw)

,C1,C2,C3,C4
DEV,2936,4232,3045,1251
TEST,4632,4570,3700,1626


[18:44:44] smoothing tuned on Madrid OOF: {'alpha': 0.5, 'size': 5}


[18:49:39] base: zero-shot 0.591 | BBSE Amsterdam class mix [0.159 0.398 0.246 0.197]


[18:54:30] noise: zero-shot 0.589 | BBSE Amsterdam class mix [0.171 0.394 0.249 0.185]


[18:54:30] (for information only, never used: true Amsterdam mix [0.291 0.339 0.26  0.111])


## 4 · Run every candidate on DEV (draw seeds 100–109)

In [7]:
DEV_SEEDS = list(range(100, 100 + N_DRAWS))
CANDS = {'E0 current': E0, 'E1 label-noise weights': V(w='noise'),
         'E2a BBSE all n': V(prior='bbse_all'), 'E2b BBSE n>=25': V(prior='bbse25'), 'E2c EM all n': V(prior='em_all'),
         'E3 agreement filter': V(agree=True), 'E4a fine-tune 10%': V(wfrac=.10), 'E4b fine-tune 50%': V(wfrac=.50),
         'E5 no smoothing': V(smooth=False)}
t = time.time(); dev = run_configs(CANDS, DEV, DEV, DEV_SEEDS, 'DEV'); dev.to_csv(OUT / 'dev_all_draws.csv', index=False)
log(f"DEV experiments finished in {(time.time() - t) / 60:.1f} min")
dev.pivot_table(index='cfg', columns='n', values='f1').round(3).assign(mean=lambda d: d.mean(1).round(3), far=dev.groupby('cfg').f1_far.mean().round(3))

[19:24:23]   [DEV] n =   5 done (1793s)


[19:59:21]   [DEV] n =  25 done (2098s)


[20:12:49]   [DEV] n =  50 done (808s)


[20:26:53]   [DEV] n = 100 done (844s)


[20:41:16]   [DEV] n = 200 done (863s)


[20:41:16] DEV experiments finished in 106.8 min


n,5,25,50,100,200,mean,far
cfg,,,,,,,
E0 current,0.639,0.693,0.728,0.747,0.766,0.715,0.467
E1 label-noise weights,0.637,0.695,0.727,0.748,0.766,0.715,0.471
E2a BBSE all n,0.627,0.686,0.717,0.733,0.752,0.703,0.462
E2b BBSE n>=25,0.639,0.686,0.717,0.733,0.752,0.705,0.464
E2c EM all n,0.619,0.693,0.728,0.747,0.766,0.711,0.463
E3 agreement filter,0.634,0.693,0.727,0.747,0.766,0.713,0.466
E4a fine-tune 10%,0.636,0.690,0.727,0.745,0.765,0.713,0.465
E4b fine-tune 50%,0.634,0.694,0.727,0.748,0.765,0.714,0.464
E5 no smoothing,0.620,0.677,0.707,0.726,0.743,0.695,0.461


## 5 · Apply the pre-registered rule

In [8]:
def compare(df, a, b):
    f = df.pivot_table(index=['n', 'seed'], columns='cfg', values='f1'); g = df.pivot_table(index=['n', 'seed'], columns='cfg', values='f1_far')
    d = f[b] - f[a]; se = d.std(ddof=1) / np.sqrt(len(d)); dfar = (g[b] - g[a]).mean()
    return d.mean(), se, dfar
rows = []
for c in CANDS:
    if c == 'E0 current': continue
    m, se, dfar = compare(dev, 'E0 current', c)
    passed = (m > 2 * se) and (dfar >= -0.005) if c != 'E5 no smoothing' else ((m > 2 * se) or (dfar > 0.005))
    rows.append({'candidate': c, 'DEV gain': round(m, 4), '2×SE': round(2 * se, 4), 'far-pixel gain': round(dfar, 4), 'decision': 'ADOPT' if passed else 'reject'})
DECIDE = pd.DataFrame(rows); DECIDE.to_csv(OUT / 'decisions.csv', index=False); display(DECIDE)

ok = set(DECIDE.query("decision == 'ADOPT'").candidate)
best = lambda names: max((n for n in names if n in ok), key=lambda n: DECIDE.set_index('candidate').loc[n, 'DEV gain'], default=None)
FINAL = dict(E0)
if 'E1 label-noise weights' in ok: FINAL['w'] = 'noise'
p = best(['E2a BBSE all n', 'E2b BBSE n>=25', 'E2c EM all n']); FINAL['prior'] = {'E2a BBSE all n': 'bbse_all', 'E2b BBSE n>=25': 'bbse25', 'E2c EM all n': 'em_all', None: 'em25'}[p]
if 'E3 agreement filter' in ok: FINAL['agree'] = True
p = best(['E4a fine-tune 10%', 'E4b fine-tune 50%']); FINAL['wfrac'] = {'E4a fine-tune 10%': .10, 'E4b fine-tune 50%': .50, None: .25}[p]
if 'E5 no smoothing' in ok: FINAL['smooth'] = False
log(f"combined configuration: {FINAL}")
if FINAL != E0:                                                       # confirm the combination on DEV with the same rule
    comb = run_configs({'E0 current': E0, 'COMBINED': FINAL}, DEV, DEV, DEV_SEEDS, 'DEV-combined'); m, se, dfar = compare(comb, 'E0 current', 'COMBINED')
    log(f"combined on DEV: gain {m:+.4f} (2SE {2*se:.4f}), far {dfar:+.4f}")
    if not ((m > 2 * se) and (dfar >= -0.005)):
        single = DECIDE.query("decision == 'ADOPT' and candidate != 'E5 no smoothing'").sort_values('DEV gain').candidate
        FINAL = CANDS[single.iloc[-1]] if len(single) else dict(E0); log(f"combination failed the rule -> fall back to {FINAL}")
json.dump({'FINAL': FINAL, 'ADOPT_E1_MADRID': ADOPT_E1_MADRID, 'SMOOTH': SMOOTH, 'BBSE': {k: v.tolist() for k, v in PRIOR_BBSE.items()}}, open(OUT / 'final_config.json', 'w'), indent=1)

,candidate,DEV gain,2×SE,far-pixel gain,decision
0,E1 label-noise weights,0.0001,0.0018,0.0039,reject
1,E2a BBSE all n,-0.0115,0.0033,-0.0050,reject
2,E2b BBSE n>=25,-0.0091,0.0023,-0.0024,reject
3,E2c EM all n,-0.0039,0.0041,-0.0037,reject
4,E3 agreement filter,-0.0009,0.0020,-0.0002,reject
5,E4a fine-tune 10%,-0.0017,0.0016,-0.0019,reject
6,E4b fine-tune 50%,-0.0009,0.0020,-0.0027,reject
7,E5 no smoothing,-0.0202,0.0013,-0.0061,reject


[20:41:16] combined configuration: {'w': 'base', 'wfrac': 0.25, 'agree': False, 'prior': 'em25', 'smooth': True}


## 6 · One-time evaluation on the untouched TEST region
Support labels come from **DEV** blocks; every scored pixel lies in **TEST** blocks. Draw seeds 0–9.

In [9]:
test = run_configs({'E0 current': E0, 'FINAL': FINAL}, DEV, TEST, list(range(N_DRAWS)), 'TEST'); test.to_csv(OUT / 'test_all_draws.csv', index=False)
T = test.groupby(['cfg', 'n']).agg(f1=('f1', 'mean'), sd=('f1', lambda x: x.std(ddof=0)), far=('f1_far', 'mean')).round(3).unstack('cfg')
T.to_csv(OUT / 'test_summary.csv'); display(T)
m, se, dfar = compare(test, 'E0 current', 'FINAL'); log(f"TEST: FINAL vs current gain {m:+.4f} ± {se:.4f} (far {dfar:+.4f})")
log(f"Madrid spatial CV for reference: current {sc_e0.mean():.3f}, label-noise {sc_e1.mean():.3f}")

[20:44:36]   [TEST] n =   5 done (200s)


[20:47:56]   [TEST] n =  25 done (200s)


[20:51:19]   [TEST] n =  50 done (203s)


[20:54:58]   [TEST] n = 100 done (218s)


[20:58:33]   [TEST] n = 200 done (216s)


f1                sd               far       
cfg E0 current  FINAL E0 current  FINAL E0 current  FINAL
n                                                        
5        0.658  0.658      0.027  0.027      0.654  0.654
25       0.706  0.706      0.010  0.010      0.705  0.705
50       0.718  0.718      0.010  0.010      0.718  0.718
100      0.730  0.730      0.007  0.007      0.734  0.734
200      0.749  0.749      0.004  0.004      0.760  0.760

[20:58:33] TEST: FINAL vs current gain +0.0000 ± 0.0000 (far +0.0000)


[20:58:33] Madrid spatial CV for reference: current 0.651, label-noise 0.652


## 7 · How to read the result
* **TEST** numbers are the honest generalisation estimate: labels and scored pixels come from different districts.
* If a candidate was rejected it goes on the *what didn't work* list, with its DEV gain and error bar.
* The submission notebook is then re-run with the frozen configuration under the organisers' protocol (draws 0–9), so the required table stays comparable.